# Evaluación automática de una versión

Este notebook recibe únicamente `model_name` y `model_version` desde el Deployment Job.

In [ ]:
import json
import tempfile
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split

dbutils.widgets.text('model_name', 'workspace.default.iris_classifier')
dbutils.widgets.text('model_version', '')
model_name = dbutils.widgets.get('model_name')
model_version = dbutils.widgets.get('model_version')
if not model_version:
    raise ValueError('model_version es obligatorio')

from iris_mlflow_utils import (
    build_deployment_config,
    build_evaluation_artifacts,
    build_probability_metrics,
    evaluate_model,
    evaluate_promotion_gate,
    load_dataset_from_spark,
)
from iris_mlflow_utils.config import build_config

deployment_config = build_deployment_config()
training_config = build_config(model_slug='random_forest')
client = MlflowClient(registry_uri='databricks-uc')
version = client.get_model_version(model_name, model_version)
model_type = version.tags.get('model_type', 'random_forest')
model_uri = f'models:/{model_name}/{model_version}'
if model_type == 'xgboost':
    model = mlflow.xgboost.load_model(model_uri)
else:
    model = mlflow.sklearn.load_model(model_uri)


In [ ]:
dataset = load_dataset_from_spark(
    spark,
    table_name=training_config.feature_table,
    table_version=training_config.feature_table_version,
    target_column=training_config.target_column,
)
x_train, x_test, y_train, y_test = train_test_split(
    dataset.features, dataset.target,
    test_size=training_config.test_size,
    random_state=training_config.random_state,
    stratify=dataset.target,
)
result = evaluate_model(model, x_test, y_test, list(range(len(dataset.classes))))
result.metrics.update(build_probability_metrics(result, y_test, list(range(len(dataset.classes)))))
metrics = {f'test_{key}': value for key, value in result.metrics.items()}
evaluation_run = mlflow.start_run(run_name=f'evaluate-{model_name.split(".")[-1]}-{model_version}')
mlflow.set_tags({'model_name': model_name, 'model_version': model_version, 'stage': 'evaluation'})
with tempfile.TemporaryDirectory() as directory:
    output_dir = Path(directory) / 'evaluation'
    paths = build_evaluation_artifacts(
        model, result, labels=list(range(len(dataset.classes))),
        class_names=dataset.classes, features=x_test, target=y_test, output_dir=output_dir
    )
    predictions = x_test.copy()
    predictions['actual'] = y_test
    predictions['prediction'] = result.predictions
    if result.probabilities is not None:
        for index, class_name in enumerate(dataset.classes):
            predictions[f'probability_{class_name}'] = result.probabilities[:, index]
    predictions.to_parquet(output_dir / 'predictions.parquet', index=False)
    (output_dir / 'metrics_summary.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    (output_dir / 'classification_report.json').write_text(json.dumps(result.report, indent=2, default=float), encoding='utf-8')
    (output_dir / 'classification_by_class.json').write_text(json.dumps(result.report, indent=2, default=float), encoding='utf-8')
    (output_dir / 'class_mapping.json').write_text(json.dumps({str(i): name for i, name in enumerate(dataset.classes)}, indent=2), encoding='utf-8')
    (output_dir / 'input_schema.json').write_text(json.dumps({str(column): str(dtype) for column, dtype in x_test.dtypes.items()}, indent=2), encoding='utf-8')
    for name, path in paths.items():
        mlflow.log_artifact(str(path), artifact_path='evaluation')
    mlflow.log_artifact(str(output_dir / 'predictions.parquet'), artifact_path='evaluation')
    mlflow.log_artifact(str(output_dir / 'metrics_summary.json'), artifact_path='evaluation')
    mlflow.log_artifact(str(output_dir / 'classification_report.json'), artifact_path='evaluation')
    mlflow.log_artifact(str(output_dir / 'classification_by_class.json'), artifact_path='evaluation')
    mlflow.log_artifact(str(output_dir / 'class_mapping.json'), artifact_path='evaluation')
    mlflow.log_artifact(str(output_dir / 'input_schema.json'), artifact_path='evaluation')
    mlflow.log_metrics(metrics)
mlflow.end_run()

champion_metrics = None
try:
    champion = client.get_model_version_by_alias(model_name, deployment_config.champion_alias)
    if champion.run_id:
        champion_metrics = client.get_run(champion.run_id).data.metrics
except Exception:
    champion_metrics = None
decision = evaluate_promotion_gate(metrics, champion_metrics, deployment_config)
client.set_model_version_tag(model_name, model_version, 'evaluation_status', 'passed' if decision.passed else 'failed')
client.set_model_version_tag(model_name, model_version, 'evaluation_decision', json.dumps(decision.as_dict()))
print(decision.as_dict())
if not decision.passed:
    raise RuntimeError(f'La versión no supera los gates: {decision.reason}')
